# Pasar formato a FAISS (Facebook AI Similarity Search)
Este archivo petrende pasar un archivo con formato .csv a faiss para optimizar la búsqueda de la información.

In [ ]:
pip install pandas langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Cargar y limpiar datos
df = pd.read_csv('base_de_datos_final.csv')

df = df.fillna("")
df = df.replace("n/a", "")

docs = []

for _, row in df.iterrows():
    texto = f"""
Nombre: {row['Nombre']} ({row['Nombre_Latin']})
Categoría: {row['Categoría']}
Constelación: {row['Constelacion']}

Datos físicos:
- Distancia: {row['Distancia']}
- Masa: {row['Masa']}
- Diámetro: {row['Diámetro']}
- Gravedad: {row['Gravedad']}
- Brillo: {row['Brillo']}

Ubicación:
{row['Ubicacion']}
Coordenadas: {row['Coordenadas']}

Descripción:
{row['Contenido']}
"""

    metadatos = {
        "id": str(row['ID']),
        "nombre": str(row['Nombre']),
        "nombre_latin": str(row['Nombre_Latin']),
        "categoria": str(row['Categoría']),
        "constelacion": str(row['Constelacion']),
        "url": str(row['URL_Wikipedia']),
    }

    docs.append(Document(
        page_content=texto.strip(),
        metadata=metadatos
    ))

# Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80
)

docs_chunked = splitter.split_documents(docs)
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base"
)

for doc in docs_chunked:
    doc.page_content = "passage: " + doc.page_content

# Crear el FAISS
vector_db = FAISS.from_documents(docs_chunked, embeddings)
# Guardar
vector_db.save_local("faiss_astro_index")

print(f"FAISS creado con {len(docs_chunked)} chunks.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

✅ FAISS creado con 1522 chunks.
